# CS3807 Deep Learning Laboratory - Part 5: Graphical Comparison Plots
Six publication-style figures comparing all experiments (saved to `results/figures/`).

In [ ]:
# Cell 1: Load metrics and histories
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from tensorflow import keras
os.makedirs('results/figures', exist_ok=True)
plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})
with open('results/all_metrics.pkl', 'rb') as f:
    saved = pickle.load(f)
metrics_df, best = saved['metrics'], saved['best_model']
hists = {}
for exp, fn in {'exp1': 'results/exp1_histories.pkl', 'exp2': 'results/exp2_histories.pkl',
                'exp3': 'results/exp3_histories.pkl', 'exp4': 'results/exp4_histories.pkl',
                'exp5': 'results/exp5_histories.pkl',
                'additional': 'results/additional_histories.pkl'}.items():
    with open(fn, 'rb') as f:
        hists[exp] = pickle.load(f)
print(f"✓ Metrics for {len(metrics_df)} models; best = {best}")

In [ ]:
# Cell 2, Fig 1: Experiment 1 - attention-type accuracy + Fig 2: Experiment 2 - position
def bar(df, title, fname, xlabel):
    d = df.sort_values('test_accuracy')
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(d['model'], d['test_accuracy'], color='steelblue')
    ax.set_xlabel(xlabel); ax.set_title(title)
    for i, v in enumerate(d['test_accuracy']):
        ax.text(v, i, f" {v:.3f}", va='center')
    fig.tight_layout(); fig.savefig(f"results/figures/{fname}", dpi=150); plt.show()
bar(metrics_df[metrics_df['experiment']=='exp1'], 'Exp 1: Attention Type @ P3 (Test Accuracy)',
    'exp1_type_accuracy.png', 'Test accuracy')
bar(metrics_df[metrics_df['experiment']=='exp2'], 'Exp 2: Attention Position (Test Accuracy)',
    'exp2_position_accuracy.png', 'Test accuracy')

In [ ]:
# Cell 3, Fig 3: Training/validation curves (Exp 1) + Fig 4: macro-F1 comparison
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, ['A0_Baseline','A1_Channel','A2_Spatial','A3_ChannelSpatial',
                                'A4_Self','A5_Combined']):
    h = hists['exp1'][name]
    ax.plot(h['accuracy'], label='train'); ax.plot(h['val_accuracy'], label='val')
    ax.set_title(name); ax.set_xlabel('epoch')
axes.flat[0].legend(); axes.flat[0].set_ylabel('accuracy')
fig.suptitle('Exp 1 Training Curves'); fig.tight_layout()
fig.savefig('results/figures/training_history_curves.png', dpi=150); plt.show()
d = metrics_df.sort_values('macro_f1')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(d['model'], d['macro_f1'], color='seagreen')
ax.set_xlabel('Macro F1'); ax.set_title('Macro-F1: All Models')
fig.tight_layout(); fig.savefig('results/figures/f1_comparison.png', dpi=150); plt.show()

In [ ]:
# Cell 4, Fig 5: Experiment 4 order ablation + Fig 6: Experiment 5 kernel ablation
bar(metrics_df[metrics_df['experiment']=='exp4'], 'Exp 4: Order Ablation @ P3 (Test Accuracy)',
    'order_ablation_exp4.png', 'Test accuracy')
bar(metrics_df[metrics_df['experiment']=='exp5'], 'Exp 5: Spatial Kernel Ablation @ P3',
    'kernel_ablation_exp5.png', 'Test accuracy')
print("✓ All 6 Part-5 figures saved to results/figures/")

In [ ]:
# Cell 5: Loss curves + Accuracy-vs-Params + MacroF1-vs-InferenceTime
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for ax, name in zip(axes.flat, ['A0_Baseline','A1_Channel','A2_Spatial','A3_ChannelSpatial',
                                'A4_Self','A5_Combined']):
    h = hists['exp1'][name]
    ax.plot(h['loss'], label='train'); ax.plot(h['val_loss'], label='val')
    ax.set_title(name); ax.set_xlabel('epoch')
axes.flat[0].legend(); axes.flat[0].set_ylabel('loss')
fig.suptitle('Exp 1 Training Loss Curves (Epoch vs Loss)'); fig.tight_layout()
fig.savefig('results/figures/training_loss_curves.png', dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(metrics_df['params'], metrics_df['test_accuracy'])
for _, r in metrics_df.iterrows():
    ax.annotate(r['model'], (r['params'], r['test_accuracy']), fontsize=7)
ax.set_xlabel('Model parameters'); ax.set_ylabel('Test accuracy')
ax.set_title('Accuracy vs Model Parameters')
fig.tight_layout(); fig.savefig('results/figures/accuracy_vs_params.png', dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(metrics_df['infer_ms_per_sample'], metrics_df['macro_f1'])
for _, r in metrics_df.iterrows():
    ax.annotate(r['model'], (r['infer_ms_per_sample'], r['macro_f1']), fontsize=7)
ax.set_xlabel('Inference time (ms/sample)'); ax.set_ylabel('Macro F1')
ax.set_title('Macro-F1 vs Inference Time')
fig.tight_layout(); fig.savefig('results/figures/f1_vs_infertime.png', dpi=150); plt.show()
print("✓ Loss + cost plots saved")

In [ ]:
# Cell 6: Additional visualizations - Sample images from all 5 classes + Per-class accuracy + Parameter count
# Load data for sample images
with open('results/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
X_test, y_test, class_names = splits['X_test'], splits['y_test'], splits['class_names']
y_true = np.argmax(y_test, axis=1)

# Vis 7: Sample input images from all 5 classes
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for idx, class_name in enumerate(class_names):
    class_idx = list(class_names).index(class_name)
    sample_indices = np.where(y_true == class_idx)[0]
    sample_img = X_test[sample_indices[0]]
    axes[idx].imshow(sample_img.astype(np.uint8))
    axes[idx].set_title(class_name)
    axes[idx].axis('off')
fig.suptitle('Sample Input Images from All 5 Classes')
fig.tight_layout()
fig.savefig('results/figures/sample_images_all_classes.png', dpi=150)
plt.show()

# Vis 21: Per-class accuracy comparison (load models ONE AT A TIME to save memory)
with open('results/all_metrics.pkl', 'rb') as f:
    saved = pickle.load(f)
best = saved['best_model']

baseline_preds = None
best_preds = None

# Load baseline, compute predictions, then clean up
tf.keras.backend.clear_session()
baseline_model = keras.models.load_model('results/models/A0_Baseline_model.h5', compile=False)
baseline_preds = np.argmax(baseline_model.predict(X_test, batch_size=32, verbose=0), axis=1)
del baseline_model
tf.keras.backend.clear_session()

# Load best model, compute predictions, then clean up
best_model = keras.models.load_model(f"results/models/{best}_model.h5", compile=False)
best_preds = np.argmax(best_model.predict(X_test, batch_size=32, verbose=0), axis=1)
del best_model
tf.keras.backend.clear_session()

# Compute per-class accuracy
per_class_acc = []
for class_name in class_names:
    class_idx = list(class_names).index(class_name)
    class_mask = y_true == class_idx
    baseline_acc = accuracy_score(y_true[class_mask], baseline_preds[class_mask])
    best_acc = accuracy_score(y_true[class_mask], best_preds[class_mask])
    per_class_acc.append({'class': class_name, 'baseline': baseline_acc, 'best': best_acc})

per_class_df = pd.DataFrame(per_class_acc)
x = np.arange(len(class_names))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, per_class_df['baseline'], width, label='Baseline', color='steelblue')
ax.bar(x + width/2, per_class_df['best'], width, label=best, color='seagreen')
ax.set_xlabel('Class')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy Comparison (Baseline vs Best)')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim([0, 1])
fig.tight_layout()
fig.savefig('results/figures/per_class_accuracy_comparison.png', dpi=150)
plt.show()

# Vis 22: Parameter count comparison (bar chart) - uses metrics_df from Cell 2
fig, ax = plt.subplots(figsize=(12, 6))
params_sorted = metrics_df.sort_values('params')
ax.barh(params_sorted['model'], params_sorted['params'], color='coral')
ax.set_xlabel('Number of Parameters')
ax.set_title('Model Parameter Count Comparison')
for i, v in enumerate(params_sorted['params']):
    ax.text(v, i, f' {v:,}', va='center')
fig.tight_layout()
fig.savefig('results/figures/parameter_count_comparison.png', dpi=150)
plt.show()

print("✓ Additional visualizations saved:")
print("  - sample_images_all_classes.png")
print("  - per_class_accuracy_comparison.png")
print("  - parameter_count_comparison.png")